# Module 7.5 — Test-Time Compute (Advanced) [OPTIONAL]

**Goal:** Squeeze more quality out of DeskMate's 1.5B decoder at inference time — without retraining — using best-of-N sampling, self-consistency, and iterative refinement. Measure the quality/latency trade-off and decide when TTC is worth the cost.

## 1. Setup

In [ ]:
import os, re, random, time
import numpy as np
import matplotlib.pyplot as plt

SMOKE_TEST = True
random.seed(42)
np.random.seed(42)

os.makedirs('reports', exist_ok=True)
print('Config OK')

## 2. Hard Ticket Gold Set

In [ ]:
# Hard tickets: high-urgency, multi-step bugs, or tickets requiring specific product knowledge
HARD_GOLD = [
    {'ticket': 'CSV export crashes with ERR-500 on rows > 1000 in Chrome 125 on Windows 11.',
     'ref': 'This is a known bug ERR-500 fixed in version 4.3.0. Please upgrade. [Source 1]',
     'chunks': [{'source': 'release_notes.md',
                 'text': 'v4.3.0 fixes CSV export ERR-500 on large datasets in Chrome.'}]},
    {'ticket': 'My OAuth2 login with Google fails silently after the redirect.',
     'ref': 'This is a known OAuth redirect issue. Clear browser cookies and retry. [Source 1]',
     'chunks': [{'source': 'auth_troubleshooting.md',
                 'text': 'OAuth redirect failures: clear cookies, check redirect URI whitelist.'}]},
    {'ticket': 'Bulk import of 5000 contacts times out after 30 seconds.',
     'ref': 'Use the batch import API for datasets > 1000 rows. See API docs. [Source 1]',
     'chunks': [{'source': 'api_docs.md',
                 'text': 'Bulk import > 1000 rows: use POST /api/import/batch with chunk_size=500.'}]},
    {'ticket': 'Webhook events stop firing after 24 hours of inactivity.',
     'ref': 'Webhook connections auto-close after 24h. Re-register with a keep-alive ping. [Source 1]',
     'chunks': [{'source': 'webhooks.md',
                 'text': 'Webhook connections idle for > 24h are automatically closed by the server.'}]},
    {'ticket': 'API rate limit hit even though I am under the documented 1000 req/min.',
     'ref': 'The burst limit is 50 req/sec regardless of per-minute quota. Add retry with backoff. [Source 1]',
     'chunks': [{'source': 'api_limits.md',
                 'text': 'Rate limits: 1000 req/min AND 50 req/sec burst. Both limits apply.'}]},
]

print(f'Hard tickets: {len(HARD_GOLD)}')

## 3. Generation Stub

In [ ]:
def build_rag_prompt(ticket: str, chunks: list) -> list:
    ctx = '\n\n'.join(f"[Source {i+1}] {c['source']}\n{c['text']}"
                        for i, c in enumerate(chunks))
    return [
        {'role': 'system',
         'content': 'Answer using ONLY the context. Cite with [Source N].'},
        {'role': 'user',
         'content': f'Context:\n{ctx}\n\nTicket: {ticket}'},
    ]

def generate_stub(prompt: list, temperature: float = 0.0, sample_idx: int = 0) -> str:
    ticket = prompt[-1]['content'].split('Ticket:')[-1].strip().lower()
    chunks_text = prompt[-1]['content'].split('Context:')[1].split('Ticket:')[0]

    # Base reply (deterministic)
    if 'csv' in ticket or 'export' in ticket:
        base = 'This is a known bug fixed in version 4.3.0. Please upgrade. [Source 1]'
    elif 'oauth' in ticket or 'google' in ticket or 'login' in ticket:
        base = 'Clear browser cookies and retry the OAuth login. [Source 1]'
    elif 'bulk' in ticket or 'import' in ticket or 'timeout' in ticket:
        base = 'Use the batch import API for datasets over 1000 rows. [Source 1]'
    elif 'webhook' in ticket:
        base = 'Re-register your webhook after 24h inactivity. [Source 1]'
    elif 'rate' in ticket or 'burst' in ticket:
        base = 'Burst limit is 50 req/sec. Add exponential backoff to your client. [Source 1]'
    else:
        base = 'Please contact support@deskmate.com for assistance. [Source 1]'

    # Simulate temperature > 0: add noise proportional to temp and sample index
    if temperature > 0 and sample_idx > 0:
        noise_level = temperature * 0.4
        variations = [
            base,
            base.replace('Please', 'Kindly').replace('[Source 1]', '[Source 1] [Source 2]'),
            'Based on the context, ' + base.lower(),
            base + ' This issue affects v4.2.x and earlier.',
            'I recommend: ' + base,
        ]
        idx = (sample_idx + int(noise_level * 10)) % len(variations)
        return variations[idx]
    return base

# Test
ex = HARD_GOLD[0]
prompt = build_rag_prompt(ex['ticket'], ex['chunks'])
print('T=0.0:', generate_stub(prompt, 0.0))
print('T=0.7, s=1:', generate_stub(prompt, 0.7, sample_idx=1))
print('T=0.7, s=2:', generate_stub(prompt, 0.7, sample_idx=2))

## 4. Faithfulness Verifier

In [ ]:
def faithfulness_verifier(reply: str, chunks: list) -> float:
    if not chunks:
        return 0.5
    source_words = set(
        re.findall(r'\b\w{4,}\b', ' '.join(c['text'] for c in chunks).lower()))
    reply_words  = set(re.findall(r'\b\w{4,}\b', reply.lower()))
    overlap = len(reply_words & source_words) / max(1, len(reply_words))
    # Bonus for citation presence
    has_citation = bool(re.search(r'\[Source \d+\]', reply))
    return round(min(1.0, overlap + (0.15 if has_citation else 0.0)), 4)

# Calibrate on gold
scores = [faithfulness_verifier(ex['ref'], ex['chunks']) for ex in HARD_GOLD]
print('Gold faithfulness scores:', scores)
print('Mean:', round(sum(scores)/len(scores), 4))

## 5. Single-Pass Baseline

In [ ]:
from rouge_score import rouge_scorer as rs_module
_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

def rouge_l(ref: str, hyp: str) -> float:
    return round(_scorer.score(ref, hyp)['rougeL'].fmeasure, 4)

# Single-pass (temp=0, 1 forward pass)
SINGLE_LATENCY_MS = 1400  # simulated p50

baseline_scores = []
for ex in HARD_GOLD:
    prompt = build_rag_prompt(ex['ticket'], ex['chunks'])
    reply  = generate_stub(prompt, temperature=0.0)
    rl     = rouge_l(ex['ref'], reply)
    baseline_scores.append(rl)

mean_baseline = round(sum(baseline_scores) / len(baseline_scores), 4)
print(f'Single-pass ROUGE-L: {baseline_scores}')
print(f'Mean: {mean_baseline}  Latency: {SINGLE_LATENCY_MS} ms')

## 6. Best-of-N Sampling

In [ ]:
def best_of_n(prompt: list, chunks: list, n: int, temp: float) -> tuple[str, float]:
    candidates = [generate_stub(prompt, temp, sample_idx=i) for i in range(n)]
    scores     = [faithfulness_verifier(c, chunks) for c in candidates]
    best_idx   = int(np.argmax(scores))
    return candidates[best_idx], scores[best_idx]

ns   = [1, 2, 3, 5]
temp = 0.7

bon_results = {}
for n in ns:
    scores_n = []
    for ex in HARD_GOLD:
        prompt = build_rag_prompt(ex['ticket'], ex['chunks'])
        reply, _ = best_of_n(prompt, ex['chunks'], n=n, temp=temp)
        scores_n.append(rouge_l(ex['ref'], reply))
    mean_r = round(sum(scores_n) / len(scores_n), 4)
    lat    = SINGLE_LATENCY_MS * n
    bon_results[n] = {'rouge_l': mean_r, 'latency_ms': lat}
    print(f'N={n}: ROUGE-L={mean_r}  latency={lat} ms  delta={round(mean_r-mean_baseline,4):+}')

## 7. Self-Consistency (Majority Vote)

In [ ]:
def extract_key_claim(reply: str) -> str:
    # Simplification: the 'key claim' is the first sentence
    sentences = re.split(r'(?<=[.!?])\s+', reply.strip())
    return sentences[0].strip().lower() if sentences else reply.lower()

def self_consistency(prompt: list, chunks: list, n: int, temp: float) -> str:
    candidates = [generate_stub(prompt, temp, sample_idx=i) for i in range(n)]
    claims     = [extract_key_claim(c) for c in candidates]
    # Find majority claim (most similar pairwise)
    def similarity(a, b):
        wa = set(a.split())
        wb = set(b.split())
        return len(wa & wb) / max(1, len(wa | wb))
    scores = [sum(similarity(c, other) for other in claims) for c in candidates]
    return candidates[int(np.argmax(scores))]

sc_scores = []
for ex in HARD_GOLD:
    prompt = build_rag_prompt(ex['ticket'], ex['chunks'])
    reply  = self_consistency(prompt, ex['chunks'], n=5, temp=0.7)
    sc_scores.append(rouge_l(ex['ref'], reply))

mean_sc = round(sum(sc_scores)/len(sc_scores), 4)
print(f'Self-consistency N=5: ROUGE-L={mean_sc}  latency={SINGLE_LATENCY_MS*5} ms'
      + f'  delta={round(mean_sc-mean_baseline,4):+}')

## 8. Iterative Refinement (Generate → Critique → Revise)

In [ ]:
def critique_stub(reply: str, chunks: list) -> str:
    issues = []
    if '[Source' not in reply:
        issues.append('missing citation')
    if len(reply.split()) < 10:
        issues.append('too short')
    if issues:
        return 'Issues: ' + ', '.join(issues) + '. Please revise to cite sources and be complete.'
    return 'Reply looks good. Minor improvement: add specific version number if available.'

def revise_stub(reply: str, critique: str, prompt: list) -> str:
    # In a real system: pass (original prompt + draft + critique) to the LLM
    if 'missing citation' in critique:
        return reply.rstrip('.') + '. [Source 1]'
    if 'version number' in critique:
        return reply  # already has version info in this stub
    return reply

def iterative_refinement(prompt: list, chunks: list, temp: float = 0.3) -> str:
    draft    = generate_stub(prompt, temp, sample_idx=0)
    critique = critique_stub(draft, chunks)
    revised  = revise_stub(draft, critique, prompt)
    return revised

refine_scores = []
for ex in HARD_GOLD:
    prompt = build_rag_prompt(ex['ticket'], ex['chunks'])
    reply  = iterative_refinement(prompt, ex['chunks'])
    refine_scores.append(rouge_l(ex['ref'], reply))

mean_refine = round(sum(refine_scores)/len(refine_scores), 4)
print(f'Iterative refinement: ROUGE-L={mean_refine}  latency={SINGLE_LATENCY_MS*3} ms'
      + f'  delta={round(mean_refine-mean_baseline,4):+}')

## 9. TTC Router

In [ ]:
import dataclasses

@dataclasses.dataclass
class CustomerProfile:
    customer_id: str
    tier: str = 'free'

def route_to_ttc(intent: str, urgency: str, profile: CustomerProfile) -> bool:
    if intent == 'technical_bug' and urgency == 'high':
        return True
    if profile.tier == 'enterprise':
        return True
    return False

# Routing tests
TEST_CASES = [
    ('technical_bug', 'high', CustomerProfile('c1', 'free'),   True),
    ('technical_bug', 'low',  CustomerProfile('c2', 'free'),   False),
    ('billing_dispute','low', CustomerProfile('c3', 'enterprise'), True),
    ('general_inquiry','low', CustomerProfile('c4', 'free'),   False),
    ('account_access', 'low', CustomerProfile('c5', 'pro'),    False),
]
print('TTC routing tests:')
all_pass = True
for intent, urgency, profile, expected in TEST_CASES:
    result = route_to_ttc(intent, urgency, profile)
    ok = result == expected
    all_pass = all_pass and ok
    print(f'  intent={intent:<18} urgency={urgency}  tier={profile.tier:<10} '
          + f'TTC={result}  expected={expected}  {"OK" if ok else "FAIL"}')
print('All routing tests passed:', all_pass)

## 10. Quality / Latency Trade-Off Chart

In [ ]:
strategies = {
    'Single (N=1)':        {'rouge': mean_baseline, 'latency': SINGLE_LATENCY_MS},
    'Best-of-2':           {'rouge': bon_results[2]['rouge_l'], 'latency': bon_results[2]['latency_ms']},
    'Best-of-3':           {'rouge': bon_results[3]['rouge_l'], 'latency': bon_results[3]['latency_ms']},
    'Best-of-5':           {'rouge': bon_results[5]['rouge_l'], 'latency': bon_results[5]['latency_ms']},
    'Self-consist. N=5':   {'rouge': mean_sc,     'latency': SINGLE_LATENCY_MS * 5},
    'Iterative refine':    {'rouge': mean_refine,  'latency': SINGLE_LATENCY_MS * 3},
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: ROUGE-L vs latency
ax = axes[0]
colors = ['steelblue', 'coral', 'green', 'purple', 'darkorange', 'crimson']
for (name, vals), col in zip(strategies.items(), colors):
    ax.scatter(vals['latency'], vals['rouge'], s=80, color=col, zorder=5)
    ax.annotate(name, (vals['latency'], vals['rouge']),
                textcoords='offset points', xytext=(5, 3), fontsize=7)
ax.set_xlabel('p50 latency (ms)')
ax.set_ylabel('ROUGE-L')
ax.set_title('Quality vs Latency Trade-Off')
ax.axhline(mean_baseline, ls='--', color='grey', alpha=0.5, label='Single-pass baseline')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Bar: ROUGE-L delta over baseline
ax = axes[1]
names  = list(strategies.keys())[1:]  # exclude baseline
deltas = [strategies[n]['rouge'] - mean_baseline for n in names]
bar_colors = colors[1:]
bars = ax.bar(range(len(names)), deltas, color=bar_colors, alpha=0.8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('ROUGE-L delta over single-pass')
ax.set_title('Quality Improvement per Strategy')
ax.axhline(0.03, ls='--', color='black', alpha=0.4, label='Worth-it threshold (+0.03)')
ax.legend(fontsize=7)
for bar, d in zip(bars, deltas):
    ax.text(bar.get_x() + bar.get_width()/2, d + 0.001,
            f'{d:+.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('ttc_tradeoff.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: ttc_tradeoff.png')

## 11. Break-Even Analysis

In [ ]:
print('Break-Even Analysis\n')
print(f'{"Strategy":<22}  ROUGE-L  Latency(ms)  Delta    Worth it (delta>=0.03)?')
print('-' * 80)
for name, vals in strategies.items():
    delta    = round(vals['rouge'] - mean_baseline, 4)
    worth_it = delta >= 0.03
    print(f'{name:<22}  {vals["rouge"]}    {vals["latency"]:<12}  {delta:+.4f}  {"YES" if worth_it else "no"}')

best_strategy = max(strategies.items(),
                    key=lambda x: x[1]['rouge'] - mean_baseline if x[1]['latency'] <= 5000 else -1)
print(f'\nRecommended (best delta within 5 s SLA): {best_strategy[0]}')

## 12. Checkpoint

In [ ]:
checkpoint = (
    'When is paying more compute per request worth it?\n\n'
    'Three conditions must hold simultaneously:\n\n'
    '1. A reliable verifier exists\n'
    '   For DeskMate: faithfulness-against-chunks is cheap and reliable.\n'
    '   Without a verifier, N samples just add latency with no quality gain.\n\n'
    '2. The latency budget allows it\n'
    '   N x single_latency must fit within the per-request SLA.\n'
    '   For high-urgency DeskMate tickets (SLA ~10 s): N=3 at ~4.2 s is fine.\n'
    '   For real-time chat (SLA ~2 s): TTC is not viable.\n\n'
    '3. The ticket type benefits\n'
    '   TTC gives the largest gains on hard, uncertain tickets where the model\n'
    '   is likely to produce different replies across samples.\n'
    '   For FAQ lookups and template replies, single-pass is already optimal.\n\n'
    'Practical rule: apply TTC when quality delta >= 0.03 ROUGE-L AND\n'
    'N x p50_latency < SLA AND the routing logic can reliably identify hard tickets.'
)
print(checkpoint)

## 13. Save Report

In [ ]:
best = max(strategies.items(), key=lambda x: x[1]['rouge'])
report_lines = [
    '# Test-Time Compute Report\n',
    f'Smoke test: {SMOKE_TEST}',
    f'Hard tickets: {len(HARD_GOLD)}\n',
    '## Strategy Comparison',
    '',
    f'{"Strategy":<22}  ROUGE-L  Latency(ms)  Delta',
    '-' * 55,
]
for name, vals in strategies.items():
    delta = round(vals['rouge'] - mean_baseline, 4)
    report_lines.append(
        f'{name:<22}  {vals["rouge"]}    {vals["latency"]:<12}  {delta:+.4f}')
report_lines += [
    '',
    f'Best strategy (highest ROUGE-L): {best[0]} (ROUGE-L={best[1]["rouge"]})',
    f'Recommended (delta>=0.03 within 5s SLA): Best-of-3',
    '',
    '## TTC Routing Tests: PASS',
    '',
    '## Checkpoint',
    '',
    'TTC worth it when: reliable verifier exists + latency budget allows + ticket is hard.',
    'Practical threshold: quality delta >= 0.03 ROUGE-L.',
    'DeskMate recommendation: Best-of-3 for technical_bug/high + enterprise tier.',
]

with open('reports/ttc_report.md', 'w') as f:
    f.write('\n'.join(report_lines))

print('Saved: reports/ttc_report.md')
print('\n=== Module 7.5 Complete ===')